# Pattern 05 · Code-Then-Execute

> **Guardian: an execution sandbox.**

This notebook builds the whole thing **by hand, right here** — a dumb "model"
that is just a function, and the LangGraph graph defined inline. Nothing is
imported from the project's library; read it top to bottom.

![Code-Then-Execute](../docs/diagrams/patterns/05.png)

## The threat
An agent that runs model-written code over poisoned data will run **whatever a data field told it to** — `import os; os.system(...)`.

## The idea
Make the model write one program, then **check it against an allowlist before running it**. Anything outside the allowlist (imports, `os`, `eval`) is refused before a line executes.

It runs **offline by default** (a stand-in model that obeys injections, so the
attack is visible with no API key). Set `PIP_MODE=live` + `OPENAI_API_KEY` to
use the real model.

## 0 · Setup — the tiny model and the imports

In [1]:
# --- setup: a deliberately gullible "LLM", written as a plain function ---
import os

def ask_llm(system: str, user: str, rules) -> str:
    """Our whole 'model'. No classes, no framework.

    PIP_MODE=live  -> asks the real OpenAI gpt-4o-mini (needs OPENAI_API_KEY)
    otherwise      -> an offline stand-in that OBEYS any injection it can see,
                      so we can watch the attack land without a key.

    `rules` is a list of (needle, reply) pairs: if `needle` appears in the text,
    the dumb model returns `reply` (it "fell for it"); else the last default.
    """
    if os.getenv("PIP_MODE") == "live":
        from openai import OpenAI
        msg = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        r = OpenAI().chat.completions.create(model="gpt-4o-mini", messages=msg, temperature=0)
        return r.choices[0].message.content
    text = (system + " " + user).lower()
    for needle, reply in rules[:-1]:
        if needle.lower() in text:
            return reply
    return rules[-1][1]  # default (benign) answer

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
print("ready · PIP_MODE =", os.getenv("PIP_MODE", "mock"))

ready · PIP_MODE = mock


## 1 · Without the pattern — the attack lands

One node, one context: the model's output *is* the control flow.

In [2]:
# ======================= INSECURE: exec() whatever the model wrote =======================
import io
from contextlib import redirect_stdout

POISONED_ROW = "Widget'; import os; os.system('rm -rf /')  #"
RULES = [("import os", "import os\nprint('boom: reached the shell')"),
         ("", "print('total =', 1200 + 800)")]

class State(TypedDict):
    question: str
    data: str
    ran_shell: bool
    output: str

def insecure_agent(state: State):
    code = ask_llm("Write python to answer the question over the data.",
                   state["question"] + "\ndata row: " + state["data"], RULES)
    ran_shell = "os.system" in code or "import os" in code
    buf = io.StringIO()
    try:
        with redirect_stdout(buf):
            exec(code, {})          # <-- the bug, on purpose
    except Exception as e:
        buf.write(str(e))
    return {"ran_shell": ran_shell, "output": buf.getvalue().strip()}

g = StateGraph(State)
g.add_node("agent", insecure_agent)
g.add_edge(START, "agent"); g.add_edge("agent", END)
insecure = g.compile()

r = insecure.invoke({"question": "total revenue?", "data": POISONED_ROW, "ran_shell": False, "output": ""})
print("output:", r["output"])
print("reached a shell?:", r["ran_shell"], "  <-- COMPROMISED" if r["ran_shell"] else "")

output: boom: reached the shell
reached a shell?: True   <-- COMPROMISED


## 2 · With the pattern — the attack bounces off

Same dumb model. The difference is the **shape of the graph**, built below.

In [3]:
# ======================= SECURE: allowlist, THEN run =======================
import ast, io
from contextlib import redirect_stdout

ALLOWED_NAMES = {"print", "sum", "min", "max", "len", "sorted", "rows_revenue"}

def check(code: str):
    """Return a reason to refuse, or None if the code is clean."""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"syntax error: {e.msg}"
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            return "imports are not allowed"
        if isinstance(node, ast.Attribute) and node.attr.startswith("__"):
            return "dunder access is not allowed"
        if isinstance(node, ast.Name) and node.id not in ALLOWED_NAMES and not isinstance(node.ctx, ast.Store):
            return f"name not allowed: {node.id}"
    return None

class State2(TypedDict):
    question: str
    data: str
    verdict: str
    output: str

def codegen_node(state: State2):
    # the model writes code from the QUESTION + schema, not the poisoned row
    code = ask_llm("Write python using only print() and rows_revenue (a list of ints).",
                   state["question"], [("", "print('total =', sum(rows_revenue))")])
    return {"verdict": code}   # stash the code in the state

def run_node(state: State2):
    code = "rows_revenue = [1200, 800]\n" + state["verdict"]
    reason = check(code)                      # the gate
    if reason:
        return {"verdict": "REFUSED: " + reason, "output": ""}
    buf = io.StringIO()
    with redirect_stdout(buf):
        exec(code, {"__builtins__": {"print": print, "sum": sum}})
    return {"verdict": "clean", "output": buf.getvalue().strip()}

g2 = StateGraph(State2)
g2.add_node("codegen", codegen_node)
g2.add_node("run", run_node)
g2.add_edge(START, "codegen"); g2.add_edge("codegen", "run"); g2.add_edge("run", END)
secure = g2.compile()

# even if we force a malicious program, the gate stops it:
print("clean program :", check("print(sum(rows_revenue))"))
print("evil program  :", check("import os"), "  <-- refused before running")
r = secure.invoke({"question": "total revenue?", "data": POISONED_ROW, "verdict": "", "output": ""})
print("verdict:", r["verdict"], "| output:", r["output"], "  <-- BLOCKED")

clean program : None
evil program  : imports are not allowed   <-- refused before running
verdict: clean | output: total = 2000   <-- BLOCKED


## 3 · What to remember

The program is a thing you can inspect and reject *before* it runs. A subprocess/container is the real isolation for production — this shows the allowlist idea. **Use it when** the task is computational: analytics, text-to-SQL.